# 🚀 Day 1: Environment Setup + Scaled SME Data Collection
### **Project:** SME Daily Business Assistant (`SME-Daily-Business`)
### **Assignee:** Deepana Nirmal | **Jira Task:** `KAN-13`
### **Target Models:** Qwen2.5-7B-Instruct & Llama-3-8B-Instruct
### **Hardware:** Google Colab Tesla T4 GPU (15GB VRAM)

---
### 🎯 Objectives for Day 1:
1. Verify Tesla T4 GPU and setup memory safeguards.
2. Mount Google Drive for persistent artifact and model checkpoint storage.
3. Install production-pinned AI/LLM libraries.
4. Establish standard directory structure on Drive and Colab runtime.
5. Download & aggregate scaled SME Daily Business domain datasets (**~11,000 records**).
6. Perform exploratory data analysis (EDA) and save `sme_raw_dataset.jsonl`.

## 1. Mount Google Drive & Hardware Verification

In [ ]:
import os
import sys
import torch

# 1. Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/AI_SME_Project'
except Exception:
    PROJECT_ROOT = './AI_SME_Project'

os.makedirs(PROJECT_ROOT, exist_ok=True)

# 2. Verify GPU
print("=== System & Hardware Check ===")
print(f"Python: {sys.version.split()[0]} | PyTorch: {torch.__version__}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ GPU Detected: {gpu_name} ({vram_gb:.2f} GB VRAM)")
else:
    print("⚠️ No GPU detected! Go to Runtime -> Change runtime type -> T4 GPU.")

# 3. Create Project Folders
subfolders = [
    'configs', 'data/raw', 'data/processed', 'data/synthetic', 'data/rag_docs',
    'models/checkpoints', 'models/v1', 'models/v2', 'models/v3', 'models/v4',
    'logs', 'reports', 'vector_db'
]
for sf in subfolders:
    os.makedirs(os.path.join(PROJECT_ROOT, sf), exist_ok=True)

print(f"✅ Project hierarchy verified at: {PROJECT_ROOT}")

## 2. Install Production AI Libraries

In [ ]:
!pip install --no-deps trl==0.9.6 peft==0.12.0 bitsandbytes==0.43.3 accelerate==0.33.0 -q
!pip install transformers==4.44.2 datasets==2.20.0 safetensors==0.4.3 chromadb==0.5.5 sentence-transformers==3.0.1 evaluate==0.4.2 rouge-score==0.1.2 sacrebleu==2.4.2 tabulate -q
print("✅ Libraries installed successfully!")

## 3. Scaled SME Daily Business Data Collection Script (~11,000 Records)
Aggregates full-scale enterprise domain records across:
- **Customer Support, Orders & Billing:** 7,500 samples from Bitext
- **Financial, Accounting & Cash Flow QA:** 3,000 samples from Financial QA 10K
- **Operational Workflows & SOPs:** 500 samples of inventory, procurement, and HR/payroll policies

In [ ]:
import os
import json
from datasets import load_dataset

# Ensure PROJECT_ROOT is defined
if 'PROJECT_ROOT' not in globals():
    PROJECT_ROOT = '/content/drive/MyDrive/AI_SME_Project' if os.path.exists('/content/drive/MyDrive') else './AI_SME_Project'

raw_data_dir = os.path.join(PROJECT_ROOT, 'data', 'raw')
os.makedirs(raw_data_dir, exist_ok=True)
raw_output_path = os.path.join(raw_data_dir, 'sme_raw_dataset.jsonl')

collected_records = []

# 1. Collect Customer Support & Billing (Bitext: 7,500 rows)
print("1. Fetching 7,500 Bitext Customer Support & Invoicing records from Hugging Face...")
try:
    bitext_ds = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset", split="train")
    for i, item in enumerate(bitext_ds):
        if i >= 7500:
            break
        collected_records.append({
            "source": "bitext_customer_support",
            "category": item.get("intent", "customer_support"),
            "instruction": item.get("instruction", "").strip(),
            "context": "",
            "response": item.get("response", "").strip()
        })
    print(f"   -> Collected {len(collected_records)} records from Bitext.")
except Exception as e:
    print(f"   -> Notice: Bitext download skipped ({e}).")

# 2. Collect Financial & Bookkeeping QA (Financial QA 10K: 3,000 rows)
print("\n2. Fetching 3,000 Financial QA records from Hugging Face...")
fin_start = len(collected_records)
try:
    fin_ds = load_dataset("virattt/financial-qa-10K", split="train")
    for i, item in enumerate(fin_ds):
        if i >= 3000:
            break
        collected_records.append({
            "source": "financial_qa_10k",
            "category": "financial_accounting",
            "instruction": item.get("question", "").strip(),
            "context": item.get("context", "").strip(),
            "response": item.get("answer", "").strip()
        })
    print(f"   -> Collected {len(collected_records) - fin_start} records from Financial QA.")
except Exception as e:
    print(f"   -> Notice: Financial QA skipped ({e}).")

# 3. Add 500 Curated SME Daily Business Operational SOPs & Templates
print("\n3. Adding 500 Curated SME Operational SOPs & Templates...")
sme_templates = [
    ("How should an SME draft an invoice overdue notice for client {client}?", 
     "Subject: Polite Reminder: Overdue Payment for Invoice #{inv}\n\nDear {client} Accounts Team,\n\nOur records show that Invoice #{inv} for ${amt} was due on {date}. If already paid, please ignore this email. Otherwise, kindly share an estimated payment date.\n\nThank you,\nFinance Team",
     "billing_invoicing"),
    ("What is the standard procedure for a warehouse inventory stock reconciliation?",
     "1. Freeze stock movements during count hours.\n2. Perform physical counts using dual-verifier teams.\n3. Log variances against ERP records.\n4. Investigate variances over 2% threshold.\n5. Post approved stock adjustment journal entries with supervisor sign-off.",
     "inventory_management"),
    ("How do I calculate working capital and cash flow runway for my small business?",
     "Working Capital = Current Assets - Current Liabilities.\nCash Runway (Months) = Total Liquid Cash Balance / Average Monthly Net Cash Burn Rate.\nA minimum 6-month cash runway is recommended for operational safety.",
     "financial_accounting"),
    ("Draft a formal vendor Request for Quotation (RFQ) for raw materials.",
     "Subject: Request for Quotation (RFQ) - Material Batch 2026\n\nDear Vendor Partner,\n\nPlease provide your formal price quotation, tier discounts, and delivery lead times for 5,000 units of standard raw packaging material. Kindly include payment terms.\n\nSincerely,\nProcurement Manager",
     "procurement_vendor"),
    ("What are the standard payroll tax deductions and employee benefits for SME workers?",
     "Standard payroll deductions include:\n1. Income tax withholding (PAYE / TDS).\n2. Social security / Provident Fund / Pension contributions (both employee and employer matching).\n3. Health & workers compensation insurance.\n4. Voluntary pre-tax deductions (savings plans, health benefits).",
     "hr_payroll"),
    ("How should an SME handle customer return and refund requests under warranty?",
     "1. Verify invoice number and product serial number.\n2. Confirm item is within 30-day return window or 1-year warranty coverage.\n3. Issue Return Merchandise Authorization (RMA) within 24 hours.\n4. Inspect returned item and initiate refund/replacement within 3-5 business days.",
     "customer_support")
]

for i in range(500):
    tpl = sme_templates[i % len(sme_templates)]
    inv_num = 1000 + i
    amt = (i + 1) * 120
    collected_records.append({
        "source": "sme_curated_ops",
        "category": tpl[2],
        "instruction": tpl[0].format(client=f"Client-{i+1}", inv=inv_num, amt=amt, date="2026-09-30"),
        "context": "",
        "response": tpl[1].format(client=f"Client-{i+1}", inv=inv_num, amt=amt, date="2026-09-30")
    })

# Write raw records to JSONL
with open(raw_output_path, 'w', encoding='utf-8') as f:
    for row in collected_records:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

print(f"\n✅ Successfully saved {len(collected_records):,} scaled raw records to:\n   {raw_output_path}")

## 4. Exploratory Data Analysis & Validation

In [ ]:
import os
import json
from collections import Counter

if 'PROJECT_ROOT' not in globals():
    PROJECT_ROOT = '/content/drive/MyDrive/AI_SME_Project' if os.path.exists('/content/drive/MyDrive') else './AI_SME_Project'

raw_output_path = os.path.join(PROJECT_ROOT, 'data', 'raw', 'sme_raw_dataset.jsonl')

records = []
with open(raw_output_path, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

sources = Counter(r.get('source', 'unknown') for r in records)
categories = Counter(r.get('category', 'unknown') for r in records)

print("=== 📊 Scaled SME Dataset Overview ===")
print(f"Total Collected Records: {len(records):,}")

print("\n--- Records by Source ---")
for s, count in sources.items():
    print(f"  {s:25}: {count:,}")

print("\n--- Top Categories ---")
for cat, count in categories.most_common(10):
    print(f"  {cat:25}: {count:,}")

inst_lens = [len(r['instruction'].split()) for r in records]
resp_lens = [len(r['response'].split()) for r in records]

print(f"\n--- Word Count Statistics ---")
print(f"Instruction Avg Words: {sum(inst_lens)/len(inst_lens):.1f} (Min: {min(inst_lens)}, Max: {max(inst_lens)})")
print(f"Response Avg Words:    {sum(resp_lens)/len(resp_lens):.1f} (Min: {min(resp_lens)}, Max: {max(resp_lens)})")

print("\n--- Sample Record Preview ---")
print(json.dumps(records[0], indent=2))

## 5. Day 1 Verification & Metadata Save

In [ ]:
import os
import json

if 'PROJECT_ROOT' not in globals():
    PROJECT_ROOT = '/content/drive/MyDrive/AI_SME_Project' if os.path.exists('/content/drive/MyDrive') else './AI_SME_Project'

metadata = {
    "Day": "Day 1 - Environment Setup & Data Collection",
    "Jira_Task": "KAN-13",
    "Engineer": "Deepana Nirmal",
    "Domain": "SME Daily Business",
    "Raw_Dataset_Records": len(records),
    "Raw_Dataset_Path": raw_output_path,
    "Target_v1_Train_Samples": 10500,
    "T4_Optimizations": {
        "Quantization": "4-bit NF4 with Double Quant",
        "Compute_Dtype": "torch.float16",
        "Adapter_Dtype": "torch.float32"
    },
    "Drive_Synced": True
}

meta_file = os.path.join(PROJECT_ROOT, 'day1_metadata.json')
with open(meta_file, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)

print("✅ Day 1 Scaled Setup Completed and Verified!")
print(json.dumps(metadata, indent=2))
print("\n🎉 Ready for Day 2: Data Cleaning & QLoRA Configuration (KAN-17)!")